# 🌿 Probaho (প্রবাহ) — Novelty 1: Survey-Lineage Machine Learning Pipeline
### Multi-Generation Cadastral Lineage Graph Validation (CS 1940 → SA 1962 → RS 1988 → BS 2021) with XGBoost, Isolation Forest & SHAP Explainability

---

## 📌 Overview & Synthetic Data Guarantee
This notebook demonstrates the complete, end-to-end Machine Learning pipeline for **Probaho Novelty 1: Survey-Lineage Graph Validation**.
- **Dataset:** 30,000 synthetic parcel records across 4 historical survey generations.
- **Lineage Edges:** 29,065 directed parent-child splits, merges, and continuous transitions.
- **100% Synthetic Non-PII Guarantee:** All owner tokens, NIDs, mouzas, khatians, and dag numbers are algorithmically generated synthetic test fixtures.
- **Core Technologies:** `NetworkX` (Topological DAG & Mathematical Area Math) + `XGBoost` (Supervised Risk Classifier) + `Isolation Forest` (Unsupervised Outlier Detector) + `SHAP` (SHapley Additive exPlanations for Trust Reports).

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, f1_score
import shap

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("✓ Libraries successfully loaded!")

## 1. Load Synthetic Cadastral Datasets
We load the parcel nodes (`probaho_novelty1_survey_lineage_30000.csv`) and the normalized graph edges (`probaho_novelty1_lineage_edges.csv`).

In [ ]:
parcels_path = "../probaho_novelty1_survey_lineage_30000.csv"
edges_path = "../probaho_novelty1_lineage_edges.csv"

df_parcels = pd.read_csv(parcels_path)
df_edges = pd.read_csv(edges_path)

print(f"Total Parcel Records: {len(df_parcels):,}")
print(f"Total Lineage Edges:  {len(df_edges):,}")
print("\nSurvey Generation Distribution:")
print(df_parcels["survey_generation"].value_counts())
print("\nAnomaly Type Distribution (Ground Truth Labels for Validation):")
print(df_parcels["anomaly_type"].value_counts())

## 2. Build Cadastral Lineage Graph with NetworkX
We construct a **Directed Acyclic Graph (DAG)** to model historical parcel splits, merges, and generational continuity.

In [ ]:
G = nx.DiGraph()
for _, r in df_parcels.iterrows():
    G.add_node(
        r["record_id"],
        area=float(r["parcel_area_shotok"]),
        gen=r["survey_generation"],
        year=int(r["survey_year"]),
        dag=str(r["dag_no"]),
        fam=str(r["family_genealogy_id"]),
        lineage_id=str(r["lineage_id"])
    )

for _, e in df_edges.iterrows():
    G.add_edge(str(e["parent_record_id"]), str(e["child_record_id"]))

print(f"Graph Nodes: {G.number_of_nodes():,}")
print(f"Graph Edges: {G.number_of_edges():,}")
print(f"Is Valid DAG (No Cycles): {nx.is_directed_acyclic_graph(G)}")

## 3. Feature Engineering (Zero Data Leakage)
We extract area-conservation metrics, in/out degrees, parent-child deficits, and genealogy conflicts without using precomputed label fields.

In [ ]:
gen_map = {"CS": 0, "SA": 1, "RS": 2, "BS": 3}
dag_fam_counts = df_parcels.groupby(["dag_no", "survey_generation"])["family_genealogy_id"].nunique().to_dict()
dag_mouza_counts = df_parcels.groupby(["dag_no", "mouza_id", "survey_generation"])["record_id"].count().to_dict()

parent_areas = {n: sum(G.nodes[p]["area"] for p in G.predecessors(n)) for n in G.nodes()}
child_areas = {n: sum(G.nodes[s]["area"] for s in G.successors(n)) for n in G.nodes()}

features = []
for _, r in df_parcels.iterrows():
    rid = r["record_id"]
    gen = r["survey_generation"]
    area = float(r["parcel_area_shotok"])
    in_d = G.in_degree(rid)
    out_d = G.out_degree(rid)
    p_area = parent_areas[rid]
    c_area = child_areas[rid]
    
    preds = list(G.predecessors(rid))
    if preds:
        p_primary = preds[0]
        p_c_sum = child_areas[p_primary]
        p_orig = G.nodes[p_primary]["area"]
        parent_child_excess = max(0.0, p_c_sum - p_orig)
        parent_child_excess_ratio = (p_c_sum / (p_orig + 1e-4)) if p_orig > 0 else 1.0
        sibling_count = len(list(G.successors(p_primary)))
    else:
        parent_child_excess = 0.0
        parent_child_excess_ratio = 1.0
        sibling_count = 0
        
    area_diff = (area - p_area) if p_area > 0 else 0.0
    area_ratio = (area / (p_area + 1e-4)) if p_area > 0 else 1.0
    inflation_pct = max(0.0, (area - p_area) / (p_area + 1e-4) * 100.0) if (gen != "CS" and p_area > 0) else 0.0
    
    child_deficit = max(0.0, c_area - area) if out_d > 0 else 0.0
    child_ratio = (c_area / (area + 1e-4)) if (out_d > 0 and area > 0) else 1.0
    
    is_orphan = 1 if (gen != "CS" and in_d == 0) else 0
    dag_fam_n = dag_fam_counts.get((r["dag_no"], gen), 1)
    dag_mouza_n = dag_mouza_counts.get((r["dag_no"], r["mouza_id"], gen), 1)
    dag_disjoint_conflict = 1 if dag_fam_n > 1 else 0
    is_foreign_genealogy = 1 if "FOREIGN" in str(r["family_genealogy_id"]) else 0
    
    features.append({
        "record_id": rid,
        "lineage_id": r["lineage_id"],
        "survey_gen_code": gen_map[gen],
        "survey_year": int(r["survey_year"]),
        "parcel_area_shotok": area,
        "in_degree": in_d,
        "out_degree": out_d,
        "parent_total_area": p_area,
        "child_total_area": c_area,
        "area_diff": area_diff,
        "area_ratio": area_ratio,
        "inflation_pct": inflation_pct,
        "parent_child_excess": parent_child_excess,
        "parent_child_excess_ratio": parent_child_excess_ratio,
        "sibling_count": sibling_count,
        "child_deficit": child_deficit,
        "child_ratio": child_ratio,
        "is_orphan": is_orphan,
        "dag_fam_count": dag_fam_n,
        "dag_mouza_count": dag_mouza_n,
        "dag_disjoint_conflict": dag_disjoint_conflict,
        "is_foreign_genealogy": is_foreign_genealogy,
        "is_merge": 1 if in_d > 1 else 0,
        "is_split": 1 if out_d > 1 else 0,
        "target_binary": 0 if r["anomaly_type"] == "NONE" else 1,
    })

fdf = pd.DataFrame(features)
print(f"Engineered feature matrix: {fdf.shape}")

## 4. Lineage-Grouped Train/Test Split
We use `GroupShuffleSplit` by `lineage_id` to guarantee that parent and child parcels belonging to the same genealogical tree never leak across the training and testing partitions.

In [ ]:
feature_cols = [c for c in fdf.columns if c not in ["record_id", "lineage_id", "target_binary"]]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(fdf, fdf["target_binary"], groups=fdf["lineage_id"]))

X_train, y_train = fdf.iloc[train_idx][feature_cols], fdf.iloc[train_idx]["target_binary"]
X_test, y_test = fdf.iloc[test_idx][feature_cols], fdf.iloc[test_idx]["target_binary"]

print(f"Training set: {len(X_train):,} samples (Anomalies: {sum(y_train)})")
print(f"Testing set:  {len(X_test):,} samples (Anomalies: {sum(y_test)})")

## 5. Train XGBoost Supervised Risk Classifier
We train XGBoost with class-imbalance weighting (`scale_pos_weight`) to optimize for high recall on rare land fraud anomalies.

In [ ]:
scale_pos = (len(y_train) - sum(y_train)) / max(1, sum(y_train))
xgb_model = xgb.XGBClassifier(
    n_estimators=220,
    max_depth=6,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)

probs = xgb_model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC Score: {roc_auc_score(y_test, probs):.4f}")
print(f"PR-AUC Score:  {average_precision_score(y_test, probs):.4f}")

## 6. Model Evaluation: Precision, Recall & Confusion Matrix

In [ ]:
best_th = 0.70
preds = (probs >= best_th).astype(int)
print(classification_report(y_test, preds, digits=4))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4), dpi=150)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Fraud Anomaly"], yticklabels=["Normal", "Fraud Anomaly"])
plt.title("Probaho Confusion Matrix (Threshold = 0.70)")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

## 7. Train Isolation Forest (Unsupervised Baseline)
Isolation Forest is evaluated as a secondary unsupervised anomaly candidate detector when labels are unavailable.

In [ ]:
iso = IsolationForest(n_estimators=180, contamination=0.02, random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_preds = np.where(iso.predict(X_test) == -1, 1, 0)
print(f"Isolation Forest Unsupervised F1 Score: {f1_score(y_test, iso_preds):.4f}")

## 8. SHAP Explainability Engine (XAI)
Using `shap.TreeExplainer`, we decompose predictions into mathematically grounded Shapley values ($f(x) = E[f(x)] + \sum \phi_i$).
This explains to land officers and judicial magistrates **exactly why** a parcel was classified as high-risk.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
sample_X = X_test.iloc[:800]
shap_values = explainer(sample_X)

# SHAP Global Beeswarm Summary Plot
plt.figure(figsize=(10, 5), dpi=150)
shap.plots.beeswarm(shap_values, max_display=10, show=True)

# SHAP Feature Importance Ranking
plt.figure(figsize=(10, 5), dpi=150)
shap.plots.bar(shap_values, max_display=10, show=True)

## 9. Individual Parcel SHAP Waterfall Explanation
We inspect an actual high-risk anomaly parcel to visualize how area inflation and missing ancestor features pushed the risk score above the decision threshold.

In [ ]:
test_parcels = fdf.iloc[test_idx].reset_index(drop=True)
anomaly_idx = test_parcels[test_parcels["target_binary"] == 1].index[0]
sample_parcel = X_test.iloc[anomaly_idx:anomaly_idx+1]
parcel_shap = explainer(sample_parcel)

print(f"Inspecting Parcel: {test_parcels.iloc[anomaly_idx]['record_id']} (Lineage: {test_parcels.iloc[anomaly_idx]['lineage_id']})")
plt.figure(figsize=(9, 5), dpi=150)
shap.plots.waterfall(parcel_shap[0], max_display=8, show=True)